In [1]:
import time
from pathlib import Path

import pandas as pd
import requests


START_DATE = "2023-01-01"
END_DATE = "2023-12-31"

OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)

URL = "https://archive-api.open-meteo.com/v1/archive"

REGIONS = {
    "North": {"latitude": 50.6292, "longitude": 3.0573},       # Lille
    "South": {"latitude": 43.2965, "longitude": 5.3698},       # Marseille
    "East": {"latitude": 48.5734, "longitude": 7.7521},        # Strasbourg
    "West": {"latitude": 47.2184, "longitude": -1.5536},       # Nantes
    "Central": {"latitude": 48.8566, "longitude": 2.3522},     # Paris
}


def fetch_weather(region, latitude, longitude):
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": ",".join([
            "temperature_2m_mean",
            "temperature_2m_min",
            "temperature_2m_max",
            "precipitation_sum",
            "wind_speed_10m_max",
        ]),
        "timezone": "Europe/Paris",
    }

    response = requests.get(URL, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    df = pd.DataFrame(data["daily"])
    df["Region"] = region
    df["latitude"] = latitude
    df["longitude"] = longitude

    return df


def main():
    all_data = []

    for region, coords in REGIONS.items():
        print(f"Récupération météo pour la région : {region}")

        df_region = fetch_weather(
            region=region,
            latitude=coords["latitude"],
            longitude=coords["longitude"],
        )

        all_data.append(df_region)
        time.sleep(1)

    weather_daily = pd.concat(all_data, ignore_index=True)

    weather_daily["time"] = pd.to_datetime(weather_daily["time"])
    weather_daily["year"] = weather_daily["time"].dt.year
    weather_daily["month"] = weather_daily["time"].dt.month

    weather_daily["temperature_range"] = (
        weather_daily["temperature_2m_max"]
        - weather_daily["temperature_2m_min"]
    )

    weather_daily["is_cold_day"] = weather_daily["temperature_2m_mean"] < 8
    weather_daily["is_hot_day"] = weather_daily["temperature_2m_mean"] > 25

    weather_by_region = weather_daily.groupby("Region", as_index=False).agg({
        "temperature_2m_mean": "mean",
        "temperature_2m_min": "mean",
        "temperature_2m_max": "mean",
        "precipitation_sum": "sum",
        "wind_speed_10m_max": "mean",
        "temperature_range": "mean",
        "is_cold_day": "sum",
        "is_hot_day": "sum",
    })

    weather_by_region = weather_by_region.rename(columns={
        "temperature_2m_mean": "avg_temperature",
        "temperature_2m_min": "avg_min_temperature",
        "temperature_2m_max": "avg_max_temperature",
        "precipitation_sum": "total_precipitation",
        "wind_speed_10m_max": "avg_wind_speed",
        "temperature_range": "avg_temperature_range",
        "is_cold_day": "nb_cold_days",
        "is_hot_day": "nb_hot_days",
    })

    weather_daily.to_csv(OUTPUT_DIR / "weather_daily_data.csv", index=False)
    weather_by_region.to_csv(OUTPUT_DIR / "weather_by_region.csv", index=False)

    print("Fichiers créés :")
    print("data/weather_daily_data.csv")
    print("data/weather_by_region.csv")
    print(weather_by_region.head())


if __name__ == "__main__":
    main()

Récupération météo pour la région : North
Récupération météo pour la région : South
Récupération météo pour la région : East
Récupération météo pour la région : West
Récupération météo pour la région : Central
Fichiers créés :
data/weather_daily_data.csv
data/weather_by_region.csv
    Region  avg_temperature  avg_min_temperature  avg_max_temperature  \
0  Central        12.933973             8.976712            17.053973   
1     East        12.497808             8.508219            16.854795   
2    North        12.011507             8.364384            15.763836   
3    South        15.997808            12.415068            19.670685   
4     West        13.729589            10.203288            17.568767   

   total_precipitation  avg_wind_speed  avg_temperature_range  nb_cold_days  \
0                949.0       20.554521               8.077260            85   
1               1187.4       18.694521               8.346575           106   
2               1015.8       22.494521    